# Localization Analysis — Unitree go2

**Course:** Ciência de Dados — FEI Mestrado  
**Description:** Analysis of RTABMAP localization quality using one or two cameras in a go2 robot.

---

### Data Sources
| File | Description |
|------|-------------|
| `localization_log.csv` | Per-update metrics from `/rtabmap/info` and `/localization_pose` (inliers, covariance, pose, etc.) |
| `plan_log.csv` | Planned path poses from `/plan` topic, grouped by `plan_id` |

### Sections
1. **Data Loading and Data Processing** — read CSV logs from a given run folder and process all data 
2. **Path Visualization** — planned path vs actual robot trajectory  
3. **Localization Quality** — inliers, hypothesis ratio, covariance over time


### 1. **Data Loading and Data Processing**

In [28]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from scipy.stats.mstats import winsorize
from plotly.subplots import make_subplots




logs_dir = Path("localization_analysis/data/logs")

loc_paths = sorted(logs_dir.glob("logger_csv_*/localization_log.csv"))
dfs_loc    = [pd.read_csv(loc_path) for loc_path in loc_paths] 

plan_paths = sorted(logs_dir.glob("logger_csv_*/plan_log.csv"))
dfs_plan    = [pd.read_csv(plan_path) for plan_path in plan_paths] 

In [29]:
print(f' Localization Dataframes: {len(dfs_loc)}\n Plan Dataframes {len(dfs_plan)}') # Number of DF

 Localization Dataframes: 5
 Plan Dataframes 5


In [30]:
# Take the first path that NAV2 stack calculated (the plan_id = 1)
last_plans = [df[df['plan_id'] == df['plan_id'].min()] for df in dfs_plan]
last_plans

[     plan_id  pose_index       x       y
 0          1           0 -2.9519  2.2049
 1          1           1 -2.9019  2.1549
 2          1           2 -2.8519  2.1049
 3          1           3 -2.8019  2.0549
 4          1           4 -2.7519  2.0049
 ..       ...         ...     ...     ...
 144        1         144  1.7117  1.5549
 145        1         145  1.7184  1.6049
 146        1         146  1.7269  1.6549
 147        1         147  1.7370  1.7049
 148        1         148  1.7481  1.7549
 
 [149 rows x 4 columns],
      plan_id  pose_index       x       y
 0          1           0 -2.8019  2.1549
 1          1           1 -2.7519  2.1049
 2          1           2 -2.7019  2.0549
 3          1           3 -2.6519  2.0049
 4          1           4 -2.6019  1.9549
 ..       ...         ...     ...     ...
 143        1         143  1.7117  1.5549
 144        1         144  1.7184  1.6049
 145        1         145  1.7269  1.6549
 146        1         146  1.7370  1.7049
 147   

In [31]:
# Drop rows where pose was not yet received (NaN)
actuals = [df.dropna(subset=['pos_x', 'pos_y']) for df in dfs_loc]
actuals

[    timestamp_sec camera_mode  node_id  inliers  matches  inlier_ratio  \
 0    1.774891e+09      single    18705        0        0        0.0000   
 1    1.774891e+09      single    18706        0        0        0.0000   
 2    1.774891e+09      single    18707        0        0        0.0000   
 3    1.774891e+09      single    18708        0        0        0.0000   
 4    1.774891e+09      single    18709        0        0        0.0000   
 5    1.774891e+09      single    18710        0        0        0.0000   
 6    1.774891e+09      single    18711        0        0        0.0000   
 7    1.774891e+09      single    18712        0        0        0.0000   
 8    1.774891e+09      single    18713        0        0        0.0000   
 9    1.774891e+09      single    18714        0        0        0.0000   
 10   1.774891e+09      single    18715        0        0        0.0000   
 11   1.774891e+09      single    18716        0        0        0.0000   
 12   1.774891e+09      s

### 2. **Path Visualization**

In [32]:
n_runs = len(actuals)
cols = 2
rows = math.ceil(n_runs / cols)
titles = [f'Run {i+1}' for i in range(n_runs)]

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=titles,
                    shared_yaxes=False)

for i, (actual_df, last_plan) in enumerate(zip(actuals, last_plans)):
    row = i // cols + 1
    col = i % cols + 1

    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Planned path',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name=f'Run {i+1}',
        line=dict(color='tomato', width=2),
        legendgroup=f'run{i+1}'
    ), row=row, col=col)

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(
    title='Planned vs Actual Path',
    hovermode='closest',
    height=500 * rows,
    width=900
)
fig.show()


#### 2.1 **Normalizing and get the median of all plans and paths**

##### 2.1.1 **Plans**

In [33]:
def winsorize_and_resample(df, x_col='pos_x', y_col='pos_y', limits=(0.00, 0.00), n_points=500):
    pos_x = np.array(winsorize(df[x_col], limits=limits))
    pos_y = np.array(winsorize(df[y_col], limits=limits))

    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])
    arc_norm = arc / arc[-1]

    t = np.linspace(0, 1, n_points)
    fx = interp1d(arc_norm, pos_x, kind='linear')
    fy = interp1d(arc_norm, pos_y, kind='linear')
    return fx(t), fy(t)

# Plan paths (use x/y columns)
resampled_plans = [winsorize_and_resample(df, x_col='x', y_col='y') for df in last_plans]

plan_xs = np.array([p[0] for p in resampled_plans])
plan_ys = np.array([p[1] for p in resampled_plans])
median_plan_x = np.median(plan_xs, axis=0)
median_plan_y = np.median(plan_ys, axis=0)


In [34]:
fig = go.Figure()

for i, (px, py) in enumerate(resampled_plans):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Plan {i+1}',
        line=dict(color='tomato', width=1),
        opacity=0.3
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x, y=median_plan_y,
    mode='lines', name='Median plan',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Plans + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()


##### 2.1.1 **Paths**

In [ ]:
def winsorize_and_resample(df, limits=(0.00, 0.2), n_points=1000):
    # 1. Winsorize para remover outliers
    pos_x = np.array(winsorize(df['pos_x'], limits=limits))
    pos_y = np.array(winsorize(df['pos_y'], limits=limits))

    # 2. Interpolar para n_points uniformes por comprimento de arco
    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])
    arc_norm = arc / arc[-1]

    t = np.linspace(0, 1, n_points)
    fx = interp1d(arc_norm, pos_x, kind='linear')
    fy = interp1d(arc_norm, pos_y, kind='linear')
    return fx(t), fy(t)

resampled = [winsorize_and_resample(df) for df in actuals]

xs = np.array([p[0] for p in resampled])
ys = np.array([p[1] for p in resampled])

median_x = np.median(xs, axis=0)
median_y = np.median(ys, axis=0)


In [36]:
fig = go.Figure()

for i, (rx, ry) in enumerate(resampled):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name=f'Run {i+1}',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=last_plan['x'], y=last_plan['y'],
    mode='lines', name='Planned path',
    line=dict(color='royalblue', dash='dash', width=2)
))
fig.add_trace(go.Scatter(
    x=median_x, y=median_y,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

### 3. **Localization Quality**